# Tutorial 1 — Molecular Fingerprints & Tanimoto Similarity
**Author:** Himanshu Goel | [Website](https://himanshugoel.github.io) | [GitHub](https://github.com/himanshugoel)

Molecular fingerprints are binary bit-vectors that encode the presence or absence of structural features. They are the backbone of similarity search, compound clustering, and many ML models in drug discovery.

In this tutorial we cover:
- Morgan fingerprints (ECFP)
- Tanimoto similarity
- Similarity matrix heatmap
- Nearest-neighbour search in a compound library

In [ ]:
# Install dependencies (Colab)
!pip install rdkit pandas numpy matplotlib seaborn -q

## 1. Load molecules from SMILES

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs, Draw
from rdkit.Chem import rdMolDescriptors
import pandas as pd
import numpy as np

smiles_dict = {
    "Aspirin":      "CC(=O)Oc1ccccc1C(=O)O",
    "Ibuprofen":    "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "Paracetamol":  "CC(=O)Nc1ccc(O)cc1",
    "Caffeine":     "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "Morphine":     "OC1=CC=C2CC3N(CCC34CCc5c4cc(O)c(OC)c5)C2=C1",
    "Terfenadine":  "OC(c1ccc(C(c2ccccc2)(c2ccccc2)O)cc1)CCCN1CCC(CC1)C(O)(c1ccccc1)c1ccccc1",
    "Cisapride":    "CCOC(=O)c1cc2cc(OC)c(OC)cc2[nH]1",
    "Metformin":    "CN(C)C(=N)NC(=N)N",
    "Atorvastatin": "CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F)cc2)n1CCC(O)CC(O)CC(=O)O",
    "Dopamine":     "NCCc1ccc(O)c(O)c1",
}

mols = {name: Chem.MolFromSmiles(smi) for name, smi in smiles_dict.items()}
print(f"Loaded {len(mols)} molecules")
for name, mol in mols.items():
    print(f"  {name}: {mol.GetNumAtoms()} atoms, {mol.GetNumBonds()} bonds")

## 2. Generate Morgan fingerprints (ECFP4)

In [ ]:
# radius=2 → ECFP4, nBits=2048
fps = {name: AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
       for name, mol in mols.items()}

# Show bit count (density) for each molecule
for name, fp in fps.items():
    n_on = sum(fp)
    print(f"  {name:15s}: {n_on:4d} / 2048 bits set ({n_on/2048*100:.1f}%)")

## 3. Tanimoto similarity matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

names = list(fps.keys())
n = len(names)
sim_matrix = np.zeros((n, n))

for i, n1 in enumerate(names):
    for j, n2 in enumerate(names):
        sim_matrix[i, j] = DataStructs.TanimotoSimilarity(fps[n1], fps[n2])

sim_df = pd.DataFrame(sim_matrix, index=names, columns=names)
print(sim_df.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(sim_df, annot=True, fmt=".2f", cmap="Blues",
            linewidths=0.5, ax=ax, vmin=0, vmax=1,
            annot_kws={"size": 8})
ax.set_title("Tanimoto Similarity Matrix (Morgan ECFP4)", fontsize=13, pad=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("tanimoto_heatmap.png", dpi=150)
plt.show()
print("Note: Morphine and Terfenadine both cause QT prolongation yet have low similarity — structure alone isn't enough!")

## 4. Nearest-neighbour search

In [ ]:
def find_nearest(query_name, fps_dict, top_k=3):
    query_fp = fps_dict[query_name]
    sims = {name: DataStructs.TanimotoSimilarity(query_fp, fp)
            for name, fp in fps_dict.items() if name != query_name}
    ranked = sorted(sims.items(), key=lambda x: x[1], reverse=True)
    print(f"\nNearest neighbours to {query_name}:")
    for name, sim in ranked[:top_k]:
        print(f"  {name:15s} Tanimoto = {sim:.3f}")

for q in ["Aspirin", "Morphine", "Caffeine"]:
    find_nearest(q, fps)

## Key takeaways
- Morgan fingerprints encode local atomic environments at a given radius
- Tanimoto similarity of 1.0 = identical fingerprint; 0.0 = no overlap
- Structurally similar compounds cluster together — but biological activity can diverge (activity cliffs)
- High Tanimoto (>0.85) generally means very similar structure; <0.3 means structurally unrelated

## Next steps
- Tutorial 2: Lipinski Ro5 & ADMET profiling
- Tutorial 5: RDKit from scratch for more descriptor types